---
### Folder structure: Map out the folder structure (e.g. tree or Python pathlib/os.walk). Document it (in source schema doc or as notebook output).
---

In [1]:
# Code block 1: Map folder structure and file types. Run this first to get export_folder_name, export_date, source, structured_file_count.
import re
from pathlib import Path
from collections import defaultdict

# Repo root: Jupyter often runs with cwd = notebooks/, so go up one if needed
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ROOT = REPO_ROOT / "data/raw/google_fit/takeout_2026-02-21"

if not ROOT.exists():
    raise FileNotFoundError(f"Export not found: {ROOT} (cwd={Path.cwd()})")

# --- Vars for raw exports log (code block 4) ---
export_folder_name = ROOT.name
_export_date_match = re.match(r"takeout_(\d{4})[-_](\d{2})[-_](\d{2})", ROOT.name, re.I)
export_date = f"{_export_date_match.group(1)}-{_export_date_match.group(2)}-{_export_date_match.group(3)}" if _export_date_match else None
source = "Google Fit and Fitbit"

# Collect all directories and file extensions
dirs = []
ext_counts = defaultdict(int)
for path in ROOT.rglob("*"):
    if path.is_dir():
        dirs.append(path.relative_to(ROOT))
    else:
        ext = path.suffix.lower() or "(no ext)"
        ext_counts[ext] += 1

structured_file_count = ext_counts.get(".csv", 0) + ext_counts.get(".json", 0)

# Sort and print folder structure (one line per directory)
dirs_sorted = sorted(dirs, key=lambda p: (len(p.parts), str(p)))
print("Folders:")
for d in dirs_sorted:
    indent = "  " * (len(d.parts))
    print(f"{indent}{d.name}/")

# File types summary
print("\nFile types (extension -> count):")
for ext, count in sorted(ext_counts.items(), key=lambda x: -x[1]):
    print(f"  {ext}: {count}")

Folders:
  Takeout/
    Fit/
    Fitbit/
      Activities/
      All Data/
      All Sessions/
      Daily activity metrics/
      Account Changes/
      Active Zone Minutes (AZM)/
      Activity Goals/
      Atrial Fibrillation ECG/
      Atrial Fibrillation PPG/
      Biometrics/
      Commerce_GoogleData/
      Daily Readiness/
      Discover/
      Email Notifications Settings_GoogleData/
      Fitbit Premium/
      Global Export Data/
      Guided Programs/
      Health Fitness Data_GoogleData/
      Heart Rate/
      Heart Rate Variability/
      InAppNotifications_GoogleData/
      Menstrual Health/
      Mindfulness/
      Oxygen Saturation (SpO2)/
      Paired Devices/
      Physical Activity_GoogleData/
      SURVEYS_GoogleData/
      Sleep Score/
      Snore and Noise Detect/
      Social/
      Stress Journal/
      Stress Score/
      Temperature/
      User Security Data/
      Your Profile/
        utd-healthy-survey/
        utd-ill-trending-worse-survey/

File types (e

---
### Data headers: Map all data headers with path — for each structured file (CSV, JSON, etc.), list path and column/field headers. Document.
---

In [2]:
# Code block 2: Map data headers with path. Run this after Block 1 to get headers_by_path, unique_header_groups.
import time
from pathlib import Path
import csv
import json

# Reuse repo root and export root (same as Block 1)
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ROOT = REPO_ROOT / "data/raw/google_fit/takeout_2026-02-21"

t0 = time.perf_counter()
print("Block 2: Mapping data headers with path")
print(f"ROOT = {ROOT.resolve()}")
if not ROOT.exists():
    raise FileNotFoundError(f"Export not found: {ROOT} (cwd={Path.cwd()})")
print("ROOT exists, scanning for .csv and .json ...")

STRUCTURED_EXTENSIONS = (".csv", ".json")

def get_csv_headers(path: Path) -> list[str] | None:
    try:
        with open(path, newline="", encoding="utf-8", errors="replace") as f:
            reader = csv.reader(f)
            first = next(reader, None)
            return first if first else None
    except Exception as e:
        return [f"(error: {e})"]

def get_json_keys(path: Path) -> list[str] | None:
    try:
        with open(path, encoding="utf-8", errors="replace") as f:
            data = json.load(f)
        if isinstance(data, dict):
            return list(data.keys())
        if isinstance(data, list) and len(data) > 0:
            first = data[0]
            return list(first.keys()) if isinstance(first, dict) else [f"(array of {type(first).__name__})"]
        if isinstance(data, list):
            return ["(empty array)"]
        return [f"(type: {type(data).__name__})"]
    except json.JSONDecodeError as e:
        return [f"(invalid JSON: {e})"]
    except Exception as e:
        return [f"(error: {e})"]

# Collect structured file paths first (so we see count immediately)
file_paths = []
for ext in STRUCTURED_EXTENSIONS:
    for path in ROOT.rglob(f"*{ext}"):
        if path.is_file():
            file_paths.append((path, ext))
file_paths.sort(key=lambda x: (str(x[0]), x[1]))

print(f"Found {len(file_paths)} structured files\n")

# Collect headers for each file
results = []
for i, (path, ext) in enumerate(file_paths):
    rel = path.relative_to(ROOT)
    if ext == ".csv":
        headers = get_csv_headers(path)
    else:
        headers = get_json_keys(path)
    results.append((rel, ext, headers))
    if (i + 1) % 100 == 0:
        print(f"  ... processed {i + 1}/{len(file_paths)}")

# Dedupe: same folder + same headers => one row (many files share structure, e.g. one CSV per date)
# Group key: (parent_dir, ext, tuple of sorted header names)
def signature(h):
    return tuple(sorted(h)) if h else ()

groups = {}  # (parent, ext, sig) -> (example_path, headers, count)
for rel, ext, headers in results:
    parent = str(rel.parent)
    sig = signature(headers)
    key = (parent, ext, sig)
    if key not in groups:
        groups[key] = [rel, headers, 0]
    groups[key][2] += 1

# Print unique (folder, headers) with one example path and file count — human-scannable
print("\nData headers (unique by folder + headers; one example path per group)\n" + "=" * 70)
for (parent, ext, _), (example_rel, headers, count) in sorted(groups.items(), key=lambda x: (x[0][0], x[0][1])):
    print(f"\n  {parent}/  [{ext}]  ({count} file{'s' if count != 1 else ''})")
    print(f"    example: {example_rel.name}")
    if headers:
        print(f"    headers: {headers}")
    else:
        print("    headers: (none)")

# For later use: full list and deduped summary
headers_by_path = {str(r[0]): r[2] for r in results}
unique_header_groups = list(groups.items())  # (parent, ext, sig) -> (example_rel, headers, count)

elapsed = time.perf_counter() - t0
if elapsed >= 60:
    mins, secs = divmod(elapsed, 60)
    print(f"\nTotal run time: {mins:.0f} m {secs:.1f} s")
else:
    print(f"\nTotal run time: {elapsed:.1f} s")

# (Block 4 raw exports log uses vars from Block 1 and Block 3 only.)

Block 2: Mapping data headers with path
ROOT = /workspace/data/raw/google_fit/takeout_2026-02-21
ROOT exists, scanning for .csv and .json ...
Found 3878 structured files

  ... processed 100/3878
  ... processed 200/3878
  ... processed 300/3878
  ... processed 400/3878
  ... processed 500/3878
  ... processed 600/3878
  ... processed 700/3878
  ... processed 800/3878
  ... processed 900/3878
  ... processed 1000/3878
  ... processed 1100/3878
  ... processed 1200/3878
  ... processed 1300/3878
  ... processed 1400/3878
  ... processed 1500/3878
  ... processed 1600/3878
  ... processed 1700/3878
  ... processed 1800/3878
  ... processed 1900/3878
  ... processed 2000/3878
  ... processed 2100/3878
  ... processed 2200/3878
  ... processed 2300/3878
  ... processed 2400/3878
  ... processed 2500/3878
  ... processed 2600/3878
  ... processed 2700/3878
  ... processed 2800/3878
  ... processed 2900/3878
  ... processed 3000/3878
  ... processed 3100/3878
  ... processed 3200/3878
  ... 

---
### list of desired data locations: (from above, manually generated)

***steps*** 
- Takeout/Fitbit/Global Export Data/  [.json]  (1 file)
- example: exercise-0.json
- headers: ['logId', 'activityName', 'activityTypeId', 'activityLevel', 'averageHeartRate', 'calories', 'duration', 'activeDuration', 'steps', 'source', 'logType', 'manualValuesSpecified', 'heartRateZones', 'activeZoneMinutes', 'lastModified', 'startTime', 'originalStartTime', 'originalDuration', 'elevationGain', 'hasGps', 'shouldFetchDetails', 'hasActiveZoneMinutes']

- Takeout/Fitbit/Global Export Data/  [.json]  (1 file)
- example: exercise-100.json
- headers: ['logId', 'activityName', 'activityTypeId', 'activityLevel', 'averageHeartRate', 'calories', 'distance', 'distanceUnit', 'duration', 'activeDuration', 'steps', 'source', 'logType', 'manualValuesSpecified', 'heartRateZones', 'activeZoneMinutes', 'speed', 'pace', 'lastModified', 'startTime', 'originalStartTime', 'originalDuration', 'elevationGain', 'hasGps', 'shouldFetchDetails', 'hasActiveZoneMinutes']
    
- Takeout/Fitbit/Health Fitness Data_GoogleData/  [.csv]  (1 file)
- example: UserExercises_2025-10-09.csv
- headers: ['exercise_id', 'exercise_start', 'exercise_end', 'utc_offset', 'exercise_created', 'exercise_last_updated', 'activity_name', 'log_type', 'pool_length', 'pool_length_unit', 'intervals', 'distance_units', 'tracker_total_calories', 'tracker_total_steps', 'tracker_total_distance_mm', 'tracker_total_altitude_mm', 'tracker_avg_heart_rate', 'tracker_peak_heart_rate', 'tracker_avg_pace_mm_per_second', 'tracker_avg_speed_mm_per_second', 'tracker_peak_speed_mm_per_second', 'tracker_auto_stride_run_mm', 'tracker_auto_stride_walk_mm', 'tracker_swim_lengths', 'tracker_pool_length', 'tracker_pool_length_unit', 'tracker_cardio_load', 'manually_logged_total_calories', 'manually_logged_total_steps', 'manually_logged_total_distance_mm', 'manually_logged_pool_length', 'manually_logged_pool_length_unit', 'events', 'activity_type_probabilities', 'autodetected_confirmed', 'autodetected_start_timestamp', 'autodetected_end_timestamp', 'autodetected_utc_offset', 'autodetected_activity_name', 'autodetected_sensor_based_activity_name', 'deletion_reason', 'activity_label', 'suggested_start_timestamp', 'suggested_end_timestamp', 'reconciliation_status']


- Takeout/Fitbit/Paired Devices/  [.csv]  (1 file)
- example: Trackers.csv
- headers: ['tracker_id', 'date_added', 'last_sync_date_time', 'batt_level', 'hardware_rev', 'is_display_distance', 'is_display_calories', 'is_display_clock', 'is_display_flower', 'is_display_elevation', 'is_display_chatter', 'is_right_handed', 'tracker_name', 'device_type', 'on_dominant_hand', 'is_display_active_minutes', 'clock_face', 'enable_ancs', 'is_bonded', 'is_display_steps', 'alarm_update_time', 'is_display_heart_rate', 'heart_rate_tracking', 'heart_rate_tracking_update_time', 'tap_enabled', 'tap_screen', 'flick_enabled', 'flick_screen']
    
    
- Takeout/Fitbit/Physical Activity_GoogleData/  [.csv]  (66 files)
- example: live_pace_2025-10-13.csv
- headers: ['timestamp', 'steps', 'distance millimeters', 'altitude gain millimeters', 'data source']
    
- Takeout/Fitbit/Physical Activity_GoogleData/  [.csv]  (5 files)
- example: steps_2025-10-01.csv
- headers: ['timestamp', 'steps', 'data source']

***sleep***
- Takeout/Fitbit/Global Export Data/  [.json]  (5 files)
- example: sleep-2025-10-08.json
- headers: ['logId', 'dateOfSleep', 'startTime', 'endTime', 'duration', 'minutesToFallAsleep', 'minutesAsleep', 'minutesAwake', 'minutesAfterWakeup', 'timeInBed', 'efficiency', 'type', 'infoCode', 'logType', 'levels', 'mainSleep']
    

- Takeout/Fitbit/Health Fitness Data_GoogleData/  [.csv]  (1 file)
- example: UserSleepScores_2025-10-09.csv
- headers: ['sleep_id', 'sleep_score_id', 'data_source', 'score_utc_offset', 'score_time', 'overall_score', 'duration_score', 'composition_score', 'revitalization_score', 'sleep_time_minutes', 'deep_sleep_minutes', 'rem_sleep_percent', 'resting_heart_rate', 'sleep_goal_minutes', 'waso_count_long_wakes', 'waso_count_all_wake_time', 'restlessness_normalized', 'hr_below_resting_hr', 'sleep_score_created', 'sleep_score_last_updated']
    
    
- Takeout/Fitbit/Health Fitness Data_GoogleData/  [.csv]  (1 file)
- example: UserSleepStages_2025-10-09.csv
- headers: ['sleep_id', 'sleep_stage_id', 'sleep_stage_type', 'start_utc_offset', 'sleep_stage_start', 'end_utc_offset', 'sleep_stage_end', 'data_source', 'sleep_stage_created', 'sleep_stage_last_updated']


- Takeout/Fitbit/Health Fitness Data_GoogleData/  [.csv]  (1 file)
- example: UserSleeps_2025-10-09.csv
- headers: ['sleep_id', 'sleep_type', 'minutes_in_sleep_period', 'minutes_after_wake_up', 'minutes_to_fall_asleep', 'minutes_asleep', 'minutes_awake', 'minutes_longest_awakening', 'minutes_to_persistent_sleep', 'start_utc_offset', 'sleep_start', 'end_utc_offset', 'sleep_end', 'data_source', 'sleep_created', 'sleep_last_updated']


- Takeout/Fitbit/Heart Rate Variability/  [.csv]  (5 files)
- example: Respiratory Rate Summary - 2025-10-09.csv
- headers: ['timestamp', 'full_sleep_breathing_rate', 'full_sleep_standard_deviation', 'full_sleep_signal_to_noise', 'deep_sleep_breathing_rate', 'deep_sleep_standard_deviation', 'deep_sleep_signal_to_noise', 'light_sleep_breathing_rate', 'light_sleep_standard_deviation', 'light_sleep_signal_to_noise', 'rem_sleep_breathing_rate', 'rem_sleep_standard_deviation', 'rem_sleep_signal_to_noise']


- Takeout/Fitbit/Physical Activity_GoogleData/  [.csv]  (5 files)
- example: respiratory_rate_sleep_summary_2025-10-09.csv
- headers: ['timestamp', 'deep sleep stats - milli breaths per minute', 'deep sleep stats - standard deviation milli breaths per minute', 'deep sleep stats - signal to noise', 'light sleep stats - milli breaths per minute', 'light sleep stats - standard deviation milli breaths per minute', 'light sleep stats - signal to noise', 'rem sleep stats - milli breaths per minute', 'rem sleep stats - standard deviation milli breaths per minute', 'rem sleep stats - signal to noise', 'full sleep stats - milli breaths per minute', 'full sleep stats - standard deviation milli breaths per minute', 'full sleep stats - signal to noise', 'data source']


- Takeout/Fitbit/Sleep Score/  [.csv]  (1 file)
- example: sleep_score.csv
- headers: ['sleep_log_entry_id', 'timestamp', 'overall_score', 'composition_score', 'revitalization_score', 'duration_score', 'deep_sleep_in_minutes', 'resting_heart_rate', 'restlessness']
    
    
- Takeout/Fitbit/Stress Score/  [.csv]  (1 file)
- example: Stress Score.csv
- headers: ['DATE', 'UPDATED_AT', 'STRESS_SCORE', 'SLEEP_POINTS', 'MAX_SLEEP_POINTS', 'RESPONSIVENESS_POINTS', 'MAX_RESPONSIVENESS_POINTS', 'EXERTION_POINTS', 'MAX_EXERTION_POINTS', 'STATUS', 'CALCULATION_FAILED']
    
    
- Takeout/Fitbit/Temperature/  [.csv]  (5 files)
- example: Computed Temperature - 2025-10-09.csv
- headers: ['type', 'sleep_start', 'sleep_end', 'temperature_samples', 'nightly_temperature', 'baseline_relative_sample_sum', 'baseline_relative_sample_sum_of_squares', 'baseline_relative_nightly_standard_deviation', 'baseline_relative_sample_standard_deviation']
    
    
- Takeout/Fitbit/Your Profile/  [.csv]  (1 file)
- example: Profile.csv
- headers: ['id', 'full_name', 'first_name', 'last_name', 'display_name_setting', 'display_name', 'username', 'email_address', 'date_of_birth', 'child', 'country', 'state', 'city', 'timezone', 'locale', 'member_since', 'about_me', 'start_of_week', 'sleep_tracking', 'time_display_format', 'gender', 'height', 'weight', 'stride_length_walking', 'stride_length_running', 'weight_unit', 'distance_unit', 'height_unit', 'water_unit', 'glucose_unit', 'swim_unit']


***resting heart rate***
- Takeout/Fitbit/Physical Activity_GoogleData/  [.csv]  (1 file)
- example: daily_readiness.csv
- headers: ['timestamp', 'score', 'type', 'readiness level', 'sleep readiness', 'heart rate variability readiness', 'resting heart rate readiness', 'data source']

---

In [3]:
# Code block 3: Scan date range. Run this after Block 1 and Block 2 to get global_date_range, valid_range, structured_folder_count.
# This process is too big and requires multi thread to complete in a reasonable time frame. (can run for over 45min without multi thread)
# Set the number of workers based on the system you are using to run this on. (more is faster)
# After editing date_range_scan.py, restart the kernel and re-run this cell to pick up changes.

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ROOT = REPO_ROOT / "data/raw/google_fit/takeout_2026-02-21"
NUM_WORKERS = 16  # e.g. 12 on EPYC server, 6 on a modern work station, 2 on older laptop
# Debug: list of folder rel paths to scan only those; None = scan all.
DEBUG_FOLDERS = None

_script_dir = REPO_ROOT / "notebooks" / "scripts" / "02_google_fit_schema_recon"
sys.path.insert(0, str(_script_dir))
import date_range_scan as _drs

scan_result = _drs.run_date_range_parallel(
    ROOT, num_workers=NUM_WORKERS, folder_filter=DEBUG_FOLDERS
);  # semicolon suppresses Jupyter displaying the returned dict

# --- Vars for raw exports log (code block 4) ---
global_date_range = scan_result["global_date_range"]
valid_range = scan_result["valid_range"]
structured_folder_count = scan_result["structured_folder_count"]


[started at 2026-02-26T00:08:03.243928] — streaming to Jupyter OK
Block 3 (parallel): Time/date columns and date range
ROOT = /workspace/data/raw/google_fit/takeout_2026-02-21
Valid date range: 2010-01-01 .. 2026-02-21 (export date from path); dates outside ignored.
Scanning 3878 files in 29 folders (one task per file) ...
(Logging each folder when complete.)

  Takeout/Fitbit/Account Changes/
    earliest: 2025-10-09 22:08:33
    latest:   2025-10-09 22:08:33

  Takeout/Fitbit/Mindfulness/
    earliest: 2025-11-03 00:00:00
    latest:   2025-11-03 00:00:00

  Takeout/Fitbit/Menstrual Health/
    earliest: None
    latest:   None

  Takeout/Fitbit/Activity Goals/
    earliest: 2025-10-05 00:00:00
    latest:   2026-02-20 00:00:00

  Takeout/Fitbit/Atrial Fibrillation ECG/
    earliest: 2025-11-23 18:14:42
    latest:   2025-12-29 02:30:48

  Takeout/Fit/Daily activity metrics/
    earliest: 2024-01-10 00:00:00
    latest:   2026-02-20 00:00:00

  Takeout/Fitbit/Heart Rate/
    earliest

---
### Create the dataset log file: data/raw/google_fit/raw_exports.json
---

In [4]:
# Raw exports log: create or update data/raw/google_fit/raw_exports.json (machine-readable).
# Uses vars from Block 1 (export_folder_name, export_date, source, structured_file_count)
# and Block 3 (global_date_range, valid_range, structured_folder_count). Run Block 1 and Block 3 first.

import json
from datetime import datetime, timezone
from pathlib import Path

# Reuse repo root (same as Block 1)
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_GOOGLE_FIT_DIR = REPO_ROOT / "data/raw/google_fit"
LOG_FILE = RAW_GOOGLE_FIT_DIR / "raw_exports.json"

# Require Block 1 vars (when run in Jupyter they live in globals())
_g = globals()
for _name in ("export_folder_name", "export_date", "source", "structured_file_count"):
    if _name not in _g:
        raise NameError(f"Run Block 1 first (missing: {_name})")

# Build entry for current export (all values JSON-serializable)
data_start = None
data_end = None
valid_range_list = None
structured_folder_count_val = None
if "global_date_range" in _g and global_date_range is not None:
    gmin, gmax = global_date_range
    data_start = str(gmin) if gmin is not None else None
    data_end = str(gmax) if gmax is not None else None
if "valid_range" in _g and valid_range is not None:
    vmin, vmax = valid_range
    valid_range_list = [vmin, vmax] if vmax else [vmin]
if "structured_folder_count" in _g:
    structured_folder_count_val = structured_folder_count

log_updated = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
new_entry = {
    "export_folder": export_folder_name,
    "export_date": export_date,
    "source": source,
    "data_start": data_start,
    "data_end": data_end,
    "valid_range": valid_range_list,
    "folders": structured_folder_count_val,
    "files": structured_file_count,
    "log_updated": log_updated,
}

# Load existing or start with empty list
RAW_GOOGLE_FIT_DIR.mkdir(parents=True, exist_ok=True)
if LOG_FILE.exists():
    data = json.loads(LOG_FILE.read_text(encoding="utf-8"))
else:
    data = {"exports": []}
if "exports" not in data:
    data["exports"] = []

# Replace existing entry for this export or append
exports = data["exports"]
replaced = False
for i, e in enumerate(exports):
    if e.get("export_folder") == export_folder_name:
        exports[i] = new_entry
        replaced = True
        break
if not replaced:
    exports.append(new_entry)

LOG_FILE.write_text(json.dumps(data, indent=2) + "\n", encoding="utf-8")
print(f"Updated {LOG_FILE.relative_to(REPO_ROOT)}")


Updated data/raw/google_fit/raw_exports.json
